# Coding assistant generation tokens

In VS Code, select the Python kernel from this repository's `.venv/bin/python` and run all cells. Edit the next cell to select runs. Each run is read from `DATA_ROOT / "runs" / run_id`.

Total input includes cache reads and cache writes. The report includes usage from failed generation attempts when available. Each row reports usage for the model specified in the run metadata. Codex records usage per turn, while Copilot records usage per model in its session checkpoint.

In [1]:
from pathlib import Path

DATA_ROOT = Path("/Volumes/Extreme SSD/swebench-experiment-results/swebench-experiments")
RUN_IDS = ["codex-100", "copilot-100"]

In [2]:
import json
from collections import Counter

from IPython.display import Markdown, display


TOKEN_COLUMNS = ("Total input", "Cached input reads", "Cache writes", "Output")
USAGE_FIELDS = {
    "codex": ("input_tokens", "cached_input_tokens", "cache_write_input_tokens", "output_tokens"),
    "copilot": ("inputTokens", "cacheReadTokens", "cacheWriteTokens", "outputTokens"),
}


def parse_usage(assistant, usage):
    counts = Counter()
    for column, field in zip(TOKEN_COLUMNS, USAGE_FIELDS[assistant]):
        value = usage[field]
        if type(value) is not int or value < 0:
            raise ValueError(f"invalid token count for {field}")
        counts[column] = value
    return counts


def read_attempt(log_path, assistant, model):
    codex_counts = Counter()
    completed_turns = 0
    checkpoint = None

    with log_path.open(encoding="utf-8") as log:
        for line_number, line in enumerate(log, start=1):
            if not line.strip():
                continue
            try:
                event = json.loads(line)
                if assistant == "codex" and event.get("type") == "turn.completed":
                    codex_counts.update(parse_usage(assistant, event["usage"]))
                    completed_turns += 1
                elif assistant == "copilot" and event.get("type") == "session.usage_checkpoint":
                    checkpoint = event
            except (AttributeError, KeyError, TypeError, ValueError) as error:
                raise ValueError(f"line {line_number}: {error}") from error

    if assistant == "codex":
        if not completed_turns:
            raise ValueError("no turn.completed usage records")
        return codex_counts

    if checkpoint is None:
        raise ValueError("no session.usage_checkpoint records")
    models = checkpoint["data"]["accountingSnapshot"]["modelMetrics"]
    if not isinstance(models, dict) or model not in models:
        raise ValueError(f"no usage for configured model {model!r} in the final checkpoint")
    return parse_usage(assistant, models[model]["usage"])


def read_run(run_dir):
    metadata = json.loads((run_dir / "run_metadata.json").read_text(encoding="utf-8"))
    assistant = metadata["assistant"]
    model = metadata.get("model")
    if not isinstance(model, str) or not model.strip():
        raise ValueError("no model specified in run_metadata.json")
    if assistant not in USAGE_FIELDS:
        raise ValueError(f"unsupported assistant {assistant!r}")
    statuses = json.loads((run_dir / "generation_status.json").read_text(encoding="utf-8"))
    if not isinstance(statuses, list):
        raise ValueError("expected a list in generation_status.json")
    instance_ids = list(dict.fromkeys(attempt["instance_id"] for attempt in statuses))

    totals = Counter()
    counted = 0
    issues = []
    for instance_id in instance_ids:
        log_path = run_dir / instance_id / f"{assistant}.jsonl"
        try:
            counts = read_attempt(log_path, assistant, model)
        except (OSError, KeyError, TypeError, ValueError) as error:
            issues.append(f"{log_path}: {error}")
            continue
        totals.update(counts)
        counted += 1

    return assistant, model, totals, counted, len(instance_ids), issues

In [3]:
def display_table(headers, rows):
    def row(values):
        return "| " + " | ".join(str(value).replace("|", r"\|") for value in values) + " |"

    display(Markdown("\n".join([row(headers), row(["---"] * len(headers)), *(row(values) for values in rows)])))


def format_counts(counts):
    return [f"{counts[column]:,}" if counts is not None else "—" for column in TOKEN_COLUMNS]


runs_dir = DATA_ROOT.expanduser() / "runs"
summary_rows = []
usage_by_run = {}
issues = []
for run_id in dict.fromkeys(RUN_IDS):
    try:
        assistant, model, totals, counted, attempted, run_issues = read_run(runs_dir / run_id)
    except (OSError, KeyError, TypeError, ValueError) as error:
        summary_rows.append([run_id, "—", "—", "Unavailable", *format_counts(None)])
        issues.append(f"{run_id}: {error}")
        continue

    status = "Unavailable" if not counted else ("Partial" if counted < attempted else "Complete")
    summary_rows.append([run_id, model, f"{counted}/{attempted}", status, *format_counts(totals if counted else None)])
    usage_by_run[run_id] = (assistant, model, totals if counted else None, status)
    issues.extend(run_issues)
    if not attempted:
        issues.append(f"{run_id}: generation_status.json lists no attempts")

display_table(["Run ID", "Model", "Attempts counted", "Coverage", *TOKEN_COLUMNS], summary_rows)
if issues:
    display(Markdown("### Data issues"))
    print("\n".join(issues))
if not summary_rows:
    print("Set RUN_IDS in the configuration cell to report generation usage.")

| Run ID | Model | Attempts counted | Coverage | Total input | Cached input reads | Cache writes | Output |
| --- | --- | --- | --- | --- | --- | --- | --- |
| codex-100 | gpt-6.2-sol | 100/100 | Complete | 42,760,124 | 38,936,192 | 0 | 282,735 |
| copilot-100 | gpt-6.1-terra | 100/100 | Complete | 51,152,386 | 48,583,842 | 2,563,075 | 465,774 |

In [4]:
from decimal import Decimal

# Load the checked-in vendor rate snapshot in USD per million tokens.
# Works from the repository root or the notebook directory.
RATE_FILE = Path("coding-assistant-token-rates.json")
if not RATE_FILE.is_file():
    RATE_FILE = Path("notebooks") / RATE_FILE
TOKEN_RATES = json.loads(RATE_FILE.read_text(encoding="utf-8"), parse_float=Decimal)

display(Markdown(
    "### Standard token rates (USD per million)\n"
    "OpenAI API Standard short-context pricing (30-minute cache writes) and "
    "GitHub Copilot default-tier pricing."
))
display_table(
    ["Assistant", "Model", "Ordinary input", "Cache reads", "Cache writes", "Output", "Source"],
    [
        [assistant, model, *(f"${rates[key]}" for key in ("input", "cache_read", "cache_write", "output")),
         f"[Vendor pricing]({entry['source']})"]
        for assistant, models in TOKEN_RATES.items()
        for model, entry in models.items()
        for rates in (entry["usd_per_million"],)
    ],
)

### Standard token rates (USD per million)
OpenAI API Standard short-context pricing (30-minute cache writes) and GitHub Copilot default-tier pricing.

| Assistant | Model | Ordinary input | Cache reads | Cache writes | Output | Source |
| --- | --- | --- | --- | --- | --- | --- |
| codex | gpt-6.2-sol | $2.0 | $0.1 | $2.5 | $10.0 | [Vendor pricing](https://developers.openai.com/api/docs/models/gpt-6.2-sol) |
| copilot | gpt-6.1-terra | $1.0 | $0.05 | $1.25 | $5.0 | [Vendor pricing](https://docs.github.com/en/copilot/reference/copilot-billing/models-and-pricing) |

### How token counts become estimated costs

The usage report counts tokens for the model configured for each run. Total input includes ordinary input, cache reads, and cache writes.

| Category | Tokens used in the estimate |
| --- | --- |
| Ordinary input | Total input minus cache reads minus cache writes |
| Cache reads | Input tokens whose previously processed state was reused |
| Cache writes | Input tokens whose processed state was saved for later reuse |
| Output | Output tokens reported by the assistant |

For Copilot, the ordinary-input calculation corresponds to the configured model's `tokenDetails.input.tokenCount` in its usage checkpoint. For Codex, the calculation uses the input, cached-input, and cache-write counts reported on completed turns.

Each category is priced once. Multiply its token count by the vendor's USD rate per million tokens and divide by 1,000,000. The total is the sum of the four unrounded category costs. The table rounds the amounts for display.

The estimate applies the rates shown above to the recorded usage. Actual charges can vary with the vendor's context tier, cache retention, and plan.

In [5]:
def estimate_costs(totals, rates):
    cache_reads = totals["Cached input reads"]
    cache_writes = totals["Cache writes"]
    ordinary_input = totals["Total input"] - cache_reads - cache_writes
    if ordinary_input < 0:
        raise ValueError("cache reads and writes exceed total input")
    token_counts = {
        "input": ordinary_input,
        "cache_read": cache_reads,
        "cache_write": cache_writes,
        "output": totals["Output"],
    }
    return {key: tokens * Decimal(rates[key]) / 1_000_000 for key, tokens in token_counts.items()}


cost_rows = []
cost_issues = []
for run_id, (assistant, model, totals, coverage) in usage_by_run.items():
    entry = TOKEN_RATES.get(assistant, {}).get(model)
    costs = None
    if totals is not None:
        if entry is None:
            cost_issues.append(f"{run_id}: no rates for {assistant}/{model} in {RATE_FILE}.")
        else:
            try:
                costs = estimate_costs(totals, entry["usd_per_million"])
            except (KeyError, ValueError) as error:
                cost_issues.append(f"{run_id}: {error}")
    amounts = [f"${cost:,.4f}" for cost in (*costs.values(), sum(costs.values()))] if costs is not None else ["—"] * 5
    cost_rows.append([run_id, model, coverage if costs is not None else "Unavailable", *amounts])

display(Markdown(
    "### Estimated generation costs (USD)\n"
    "Estimates apply the listed rates uniformly to recorded usage. "
    "Actual charges depend on context tiers, cache retention, and plan billing. "
    "A partial estimate covers only attempts with usable logs."
))
display_table(
    ["Run ID", "Model", "Coverage", "Ordinary input", "Cache reads", "Cache writes", "Output", "Total"],
    cost_rows,
)
if cost_issues:
    print("\n".join(cost_issues))

### Estimated generation costs (USD)
Estimates apply the listed rates uniformly to recorded usage. Actual charges depend on context tiers, cache retention, and plan billing. A partial estimate covers only attempts with usable logs.

| Run ID | Model | Coverage | Ordinary input | Cache reads | Cache writes | Output | Total |
| --- | --- | --- | --- | --- | --- | --- | --- |
| codex-100 | gpt-6.2-sol | Complete | $7.6479 | $3.8936 | $0.0000 | $2.8274 | $14.3688 |
| copilot-100 | gpt-6.1-terra | Complete | $0.0055 | $2.4292 | $3.2038 | $2.3289 | $7.9674 |